# 第3章章节实践：MPI 进程扩展性

## 本节学习目标

本实践要求综合运用本章知识完成可复现的工程任务。请保留命令、参数、正确性结果和分析结论。

## 环境检查

直接检查本节需要的运行环境；若检查失败，请先在对应 CPU/NPU 节点加载课程要求的工具链。


In [ ]:
import shutil, subprocess
for tool in ("cmake", "mpicxx", "mpirun"):
    path = shutil.which(tool)
    if path is None:
        raise RuntimeError(f"缺少必需工具：{tool}")
    print(f"{tool}: {path}")


## 必要背景与实验材料

本实践基于本章 `src/` 中的课程工程副本。开始前应完成前面各小节，并能解释工程的关键源码、构建入口、正确性门槛和计时字段。

## 实践任务

1. 查询调度器或主机允许的 CPU slot
2. 选择 1/2/4/8 的可用子集运行相同矩阵
3. 记录 total、broadcast、compute、gather 和 balance ratio
4. 用最慢 rank 与显式通信解释扩展性

## 核心知识与关键源码解析

本章实践只调整 Notebook 中的运行参数和实验配置，不修改 `src/`。请保持输入与正确性阈值一致，每次只改变一个变量，并说明它如何影响数据流和性能。

## 实验记录模板

执行下面的准备命令，然后在目标环境中完成任务。不要把参考答案中的结论当作实测结果。

In [ ]:
%%bash
set -e
cd src/mpi_spmv
bash scripts/build.sh
for processes in 1 2 4; do
  MPI_PROCESSES=$processes bash scripts/run_mpi.sh \
    --matrix U1 --warmup 1 --repeat 3 \
    --csv "results/U1_${processes}p.csv"
done

## 评价标准

必须限制为环境实际允许的进程数，确认 correctness=PASS，并区分一次性 distribution 与每轮端到端时间。

## 查看参考答案

参考答案给出方法和判断依据，不提供虚构的固定性能数字。

## 预期现象与结果分析

正确性门槛应首先通过；性能结果随硬件、软件栈和系统负载变化。若修改后没有加速或出现退化，也应依据阶段计时、通信次数或资源竞争给出解释。

## 实践小结

完成报告时，应明确实验环境、唯一修改变量、正确性门槛、计时口径和观察到的限制。

## 工程实践提交物与完成标准

章测必须基于 `src/mpi_spmv/`，不得只回答概念题。操作链：构建 → 跑 CPU/1 rank 基线 → 用 2/4 rank 运行 → 检查累计 nnz 分区和 collective → 记录 compute/communication/total/error → 分析扩展性。

提交物：实际命令与环境；阅读或修改的真实文件/函数/参数；字段为“Ranks、Total、Compute、Communication、Balance、Error”的结果表；正确性判据；基于数据的结论。性能数字不作为固定答案。

完成标准：命令指向真实脚本或可执行文件，数据来自同口径运行，并能解释结果。


## 四类考核

以下四题中，客观题答案唯一，凭按 CSR 行划分计算任务的偏移语义即可判定；简单/中等/困难题基于本章实验，要求用命令、输出、CSV 或计算过程作为证据，不接受无证据的概念回答。

### 1. 客观题

（1）单选：按行数均分 1000 行给 4 个 rank（offset=[0,250,500,750,1000]），rank 2 负责的行区间是（　）

A. [0, 250)　　B. [250, 500)　　C. [500, 750)　　D. [750, 1000)

（2）判断（对/错）：分到 0 行的 rank 不参与本地 SpMV 数值计算，但仍必须参与 collective 与最终校验，否则会通信卡死。（　）

### 2. 简单题

给出 1/2/4 rank 的运行命令与结果表（Ranks、Total、Compute、Communication、Balance、Error），并指出表中哪些字段来自 rank 本地计时、哪些来自 collective/同步计时。

### 3. 中等题

计算各 rank 的局部 nnz 与 compute 时间，给出不均衡度量（如 max/avg 或 (max-min)/avg），说明行数均分但 nnz 不均时扩展性为何变差；用本次 CSV 的实际字段作证据。

### 4. 困难题

用实测数据诊断扩展性：把 total 分解为一次性 distribution、每轮 compute、每轮 communication 与同步，计算通信占比随 rank 的变化，说明最优 rank 数受什么限制，并给出下一步单变量实验设计（只改变一个变量）。
